# Agent 5: GitHub Analyzer Agent

This notebook implements the complete, detailed implementation of the **GitHub Analyzer Agent** used in CareerAtlas. The agent leverages the GitHub GraphQL API to select top repositories, retrieves repo metadata (languages, commit counts, file trees), extracts key files in a manifest-first manner, analyzes coding behavior using Groq, and runs confidence-demoting checks to ground inferred skills in concrete evidence.

### Step 1: API Keys Setup
Please configure your API keys here.

In [ ]:
import os
import getpass

if not os.environ.get("GITHUB_ACCESS_TOKEN"):
    os.environ["GITHUB_ACCESS_TOKEN"] = getpass.getpass("Enter your GitHub Access Token: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

class Settings:
    groq_api_key = os.environ.get("GROQ_API_KEY", "")
    groq_model = "llama-3.3-70b-versatile"
    github_access_token = os.environ.get("GITHUB_ACCESS_TOKEN", "")

settings = Settings()

In [ ]:
# Install required dependencies
# !pip install pydantic httpx langchain-groq

### Step 2: Imports & Pydantic Schemas
These define the inferred repositories, summaries, and skill evidence.

In [ ]:
import asyncio
from typing import List, Dict, Optional, Any
import httpx
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

class GitHubRepoInfo(BaseModel):
    name: str
    owner: str
    url: str
    description: Optional[str] = None
    stargazerCount: int
    pushedAt: Optional[str] = None
    primaryLanguage: Optional[str] = None
    isOwner: bool

class SkillEvidence(BaseModel):
    skill: str = Field(description="The skill/technology name.")
    evidence: str = Field(description="Direct quote of file path or language stat showing this skill.")
    confidence: str = Field(description="low | medium | high")

class RepoAnalysisResult(BaseModel):
    summary: str = Field(description="Brief summary of what the repo does.")
    coding_behavior: str = Field(description="Analysis of coding style and quality.")
    inferred_skills: List[SkillEvidence] = Field(description="Technical skills inferred with evidence.")

class ProfileAnalysisResult(BaseModel):
    overall_summary: str = Field(description="Aggregate technical profile summary.")
    overall_coding_behavior: str = Field(description="Coding behavior across all repositories.")

### Step 3: Manifest-First File Selection
Isolates package configuration files and entrypoints to limit token use.

In [ ]:
MANIFEST_FILES = {
    "package.json", "requirements.txt", "pyproject.toml", "go.mod", "pom.xml",
    "cargo.toml", "build.gradle", "composer.json", "gemfile", "pubspec.yaml",
    "dockerfile", "docker-compose.yml", "setup.py", "pipfile",
}
ENTRYPOINT_FILES = {"main.py", "app.py", "index.js", "index.ts", "server.ts", "server.js", "main.go", "main.rs"}
SOURCE_EXTENSIONS = {".py", ".js", ".jsx", ".ts", ".tsx", ".go", ".rs", ".java", ".cpp", ".c", ".h", ".cs"}
_SKIP_DIRS = {"node_modules", "venv", ".git", "dist", "build", ".venv"}

def select_files(file_tree: List[str], cap: int = 12) -> List[str]:
    def name(p): return p.lower().split("/")[-1]
    def skipped(p): return any(d in p.lower().split("/") for d in _SKIP_DIRS)

    tree = [f for f in file_tree if not skipped(f)]
    manifests = [f for f in tree if name(f) in MANIFEST_FILES]
    readmes = [f for f in tree if name(f).startswith("readme")]
    entrypoints = [f for f in tree if name(f) in ENTRYPOINT_FILES]
    source = [
        f for f in tree
        if any(name(f).endswith(ext) for ext in SOURCE_EXTENSIONS)
        and len(f.split("/")) <= 3
        and "test" not in name(f) and "spec" not in name(f)
    ]
    ordered, seen = [], set()
    for bucket in (manifests, readmes, entrypoints, source):
        for f in bucket:
            if f not in seen:
                seen.add(f)
                ordered.append(f)
    return ordered[:cap]

### Step 4: GitHub API Functions
REST and GraphQL wrappers to read directories and download file bytes.

In [ ]:
_REST_HEADERS = {
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "CareerAtlas-App",
    "X-GitHub-Api-Version": "2022-11-28",
}

async def fetch_authenticated_login(access_token: str) -> str:
    async with httpx.AsyncClient() as client:
        r = await client.get("https://api.github.com/user", headers={**_REST_HEADERS, "Authorization": f"Bearer {access_token}"})
        return r.json().get("login", "") if r.status_code == 200 else ""

async def fetch_languages(owner: str, repo: str, access_token: str) -> Dict[str, float]:
    async with httpx.AsyncClient() as client:
        r = await client.get(f"https://api.github.com/repos/{owner}/{repo}/languages", headers={**_REST_HEADERS, "Authorization": f"Bearer {access_token}"})
        if r.status_code != 200: return {}
        byte_counts = r.json() or {}
        total = sum(byte_counts.values())
        return {lang: round(b * 100 / total, 1) for lang, b in byte_counts.items()} if total > 0 else {}

async def fetch_commit_stats(owner: str, repo: str, login: str, access_token: str) -> Dict[str, Any]:
    async with httpx.AsyncClient() as client:
        r = await client.get(f"https://api.github.com/repos/{owner}/{repo}/commits", params={"author": login, "per_page": 100}, headers={**_REST_HEADERS, "Authorization": f"Bearer {access_token}"})
        if r.status_code != 200: return {"commit_count": 0, "first_commit_at": None, "last_commit_at": None}
        commits = r.json() or []
        if not commits: return {"commit_count": 0, "first_commit_at": None, "last_commit_at": None}
        dates = [c.get("commit", {}).get("author", {}).get("date") for c in commits]
        dates = [d for d in dates if d]
        return {"commit_count": len(commits), "first_commit_at": min(dates) if dates else None, "last_commit_at": max(dates) if dates else None}

async def fetch_github_graphql(query: str, variables: dict, access_token: str) -> Dict[str, Any]:
    async with httpx.AsyncClient() as client:
        r = await client.post("https://api.github.com/graphql", json={"query": query, "variables": variables}, headers={"Authorization": f"Bearer {access_token}", "Accept": "application/json", "User-Agent": "CareerAtlas-App"})
        if r.status_code != 200: raise Exception(f"GraphQL failed: {r.text}")
        d = r.json()
        if "errors" in d: raise Exception(f"GraphQL errors: {d['errors']}")
        return d["data"]

async def fetch_repo_file_tree(owner: str, repo: str, access_token: str) -> List[str]:
    async with httpx.AsyncClient() as client:
        r = await client.get(f"https://api.github.com/repos/{owner}/{repo}", headers={**_REST_HEADERS, "Authorization": f"Bearer {access_token}"})
        if r.status_code != 200: return []
        branch = r.json().get("default_branch", "main")
        tree_r = await client.get(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1", headers={**_REST_HEADERS, "Authorization": f"Bearer {access_token}"})
        return [item["path"] for item in tree_r.json().get("tree", []) if item["type"] == "blob"] if tree_r.status_code == 200 else []

async def fetch_file_content(owner: str, repo: str, path: str, access_token: str) -> str:
    async with httpx.AsyncClient() as client:
        r = await client.get(f"https://api.github.com/repos/{owner}/{repo}/contents/{path}", headers={"Authorization": f"Bearer {access_token}", "Accept": "application/vnd.github.v3.raw", "User-Agent": "CareerAtlas-App"})
        return r.text if r.status_code == 200 else ""

### Step 5: Grounding Filter & Repo Analyzer
Demotes ungrounded skills to low confidence and generates structured summaries using Groq.

In [ ]:
def _filter_evidence(skills: List[SkillEvidence], fetched_paths: List[str], languages: Dict[str, float]) -> List[SkillEvidence]:
    path_blob = " ".join(fetched_paths).lower()
    lang_names = {l.lower() for l in languages}
    out = []
    for s in skills:
        ev = (s.evidence or "").lower()
        backed = any(p and p in ev for p in path_blob.split()) or any(l in ev or l in s.skill.lower() for l in lang_names)
        out.append(s if backed else SkillEvidence(skill=s.skill, evidence=s.evidence, confidence="low"))
    return out

def _commit_context(stats: Dict[str, Any]) -> str:
    n = stats.get("commit_count", 0)
    if not n: return "no owner-authored commits found"
    span = f" between {stats['first_commit_at'][:10]} and {stats['last_commit_at'][:10]}" if stats.get("first_commit_at") else ""
    return f"{n}{'+' if n >= 100 else ''} owner-authored commits{span}"

REPO_ANALYSIS_PROMPT = PromptTemplate.from_template(
    """You are a Senior Software Engineer reviewing a candidate's repo.
    Repository Name: {repo_name}
    Description: {description}
    Primary Language: {language}
    Languages breakdown: {languages}
    Commit activity: {commit_context}
    Key files:
    {file_contents}
    Identify technical skills and cite direct file path or language evidence.
    """
)

async def analyze_repository(owner: str, repo: str, login: str, access_token: str) -> dict:
    file_tree, languages, commit_stats = await asyncio.gather(
        fetch_repo_file_tree(owner, repo, access_token),
        fetch_languages(owner, repo, access_token),
        fetch_commit_stats(owner, repo, login, access_token),
    )
    selected = select_files(file_tree)
    
    file_contents = ""
    for path in selected:
        content = await fetch_file_content(owner, repo, path, access_token)
        file_contents += f"\n\n--- FILE: {path} ---\n{content[:2000]}\n"
        
    model = ChatGroq(model=settings.groq_model, groq_api_key=settings.groq_api_key, temperature=0.1)
    chain = REPO_ANALYSIS_PROMPT | model.with_structured_output(RepoAnalysisResult)
    
    result: RepoAnalysisResult = await chain.ainvoke({
        "repo_name": repo,
        "description": "Candidate repo",
        "language": "Python",
        "languages": ", ".join(languages.keys()),
        "commit_context": _commit_context(commit_stats),
        "file_contents": file_contents
    })
    
    skills = _filter_evidence(result.inferred_skills, selected, languages)
    return {"summary": result.summary, "coding_behavior": result.coding_behavior, "inferred_skills": skills, "languages": languages, **commit_stats}

### Step 6: Test Execution
Trace github analysis.

In [ ]:
async def run_demo():
    login = await fetch_authenticated_login(settings.github_access_token)
    print(f"Connected as: {login}")
    result = await analyze_repository(login, "Hello-World", login, settings.github_access_token)
    print("\nSummary:\n", result["summary"])
    print("\nCoding Behavior:\n", result["coding_behavior"])
    print("\nInferred Skills:")
    for s in result["inferred_skills"]:
        print(f"- {s.skill}: confidence={s.confidence} (evidence={s.evidence})")

try:
    asyncio.run(run_demo())
except Exception as e:
    print(f"Execution skipped or failed. Error: {e}")